# **Project Name** - IndustryGPT on Financial Literacy and Personal Finance Management

# **Project Summary**

**Problem Statement:** In today's complex financial landscape, many individuals struggle to manage their personal finances effectively due to limited access to personalized financial education. This knowledge gap can result in poor financial decisions, leading to issues such as debt accumulation, inadequate savings, and insufficient retirement planning. Traditional financial advice often fails to meet the needs of those with limited financial literacy.

**Project Goal:** The aim of this project is to develop an AI-powered LLM chatbot that delivers personalized financial advice and education. Utilizing natural language processing (NLP), the chatbot will provide clear, accurate, and contextually relevant guidance on essential personal finance topics such as budgeting, saving, credit management, investment strategies, and retirement planning. By democratizing access to financial knowledge, the chatbot seeks to empower users to make informed decisions and improve their financial outcomes.

**Data Collection:** The chatbot is trained on a curated selection of reputable financial education articles and standard datasets. The data collection process involves using Optical Character Recognition (OCR) to extract text from image-based PDF documents, employing a Python script with libraries like pytesseract and pdf2image. This process compiles the extracted content into a structured format for training.

**Data Preprocessing:** The data preparation phase includes merging datasets, creating a unified training set from multiple sources, and ensuring proper formatting for training. The GPT-2 model is initially used for text generation; however, due to performance limitations, the decision was made to fine-tune the Llama-2-7B model, utilizing techniques for memory efficiency, such as QLoRA (Quantized Low-Rank Adaptation).

**Model Fine-Tuning:** The fine-tuning of the Llama-2-7B chat model is executed using a custom dataset of 1,000 samples and collected articles. Key parameters are defined to optimize training, including 4-bit precision loading, a batch size of 4, and specific configurations for LoRA. The fine-tuned model is saved for deployment, enhancing its response capabilities for financial literacy queries.

**Inference and Interaction:** The fine-tuned model is deployed to handle inference tasks, enabling the chatbot to provide insightful answers on financial topics. A web interface is created using Flask, allowing users to interact with the chatbot through a user-friendly webpage. The interface includes features for sending messages and displaying responses, along with Ngrok integration for external access.

**Conclusion:** The IndustryGPT project aims to bridge the financial literacy gap by offering an interactive AI chatbot that provides personalized financial education. Through the use of advanced NLP techniques and tailored datasets, the chatbot is designed to empower users with the knowledge needed to make informed financial decisions, ultimately improving their financial well-being.

# **GitHub Link**

**GitHub Link -** https://github.com/Satyam-G-Kulkarni/IndustryGPT---Financial-Literacy-and-Personal-Finance-Management

# **Problem Statement**

In today's complex financial landscape, many individuals struggle with managing their personal finances due to a lack of accessible, reliable, and personalized financial education. This knowledge gap can lead to poor financial decisions, resulting in debt accumulation, inadequate savings, and insufficient retirement planning. Traditional financial advice is often generalized and inaccessible to those who need it most, particularly individuals with limited financial literacy.

The goal of this project is to develop an AI-powered LLM chatbot that provides users with personalized financial advice and education. This chatbot will leverage natural language processing (NLP) to deliver clear, accurate, and contextually relevant guidance on key personal finance topics, including budgeting, saving, credit management, investment strategies, and retirement planning. By democratizing access to financial knowledge, the chatbot aims to empower users to make informed decisions and achieve better financial outcomes.

# **Let's Start**

## **Libraries and Packages**

In [ ]:
!pip install transformers==4.12.0 datasets torch datasets

### All necesary libraries

In [ ]:
!pip install pytesseract pdf2image
!apt-get install tesseract-ocr

In [ ]:
!apt-get install -y poppler-utils

In [ ]:
!pip install -q accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.40.2 transformers==4.31.0 trl==0.4.7

In [ ]:
!pip install huggingface_hub

In [ ]:
!pip install pyngrok

In [ ]:
!ngrok authtoken "2gorxOm9YlGFf8MoRgQfVzx6RqY_de56fyN9P51du4UayK37"

In [ ]:
import os
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from datasets import load_dataset
from trl import SFTTrainer

In [ ]:
torch.cuda.is_available()

## **Data Import and Cleaning**

### **Data Collection and Pre-processing**

For LLM chatbot project, I trained the model using well-established financial education articles and standard dataset. I chose these sources because they provide simplified, comprehensive, and trustworthy content that is crucial for delivering accurate financial literacy and personal finance advice. This approach allowed me to ensure the chatbot provides users with high-quality and practical guidance in line with industry best practices.

### Data Collection Code

In [ ]:
import os
import pytesseract
from pdf2image import convert_from_path

# pytesseract.pytesseract.tesseract_cmd = r'/usr/bin/tesseract'  # Path to tesseract executable

# Function to extract text from image-based PDF using OCR
def extract_text_from_image_based_pdf(pdf_file):
    text = ""
    try:
        # Convert PDF pages to images
        images = convert_from_path(pdf_file)

        # Perform OCR on each page image
        for image in images:
            page_text = pytesseract.image_to_string(image)
            text += page_text + "\f"  # Add form feed after each page
    except Exception as e:
        print(f"Error processing {pdf_file}: {e}")
    return text

# Specify the directory containing the PDFs
pdf_directory = "/content/drive/MyDrive/Projects/Deep Learning for NLP/Data"

# List to hold the final cleaned texts of all articles
processed_articles = []

# Iterate through all files in the directory
for filename in os.listdir(pdf_directory):
    if filename.endswith(".pdf"):
        pdf_path = os.path.join(pdf_directory, filename)

        # Extract text from the current image-based PDF
        extracted_text = extract_text_from_image_based_pdf(pdf_path)

        # Store the extracted text for further processing
        processed_articles.append({
            'filename': filename,
            'extracted_text': extracted_text
        })

        print(f"Processed: {filename}")

# Function to save combined text to a .txt file
def save_combined_text(processed_articles, output_file):
    with open(output_file, "w") as file:
        for article in processed_articles:
            file.write(f"--- Start of {article['filename']} ---\n")
            file.write(article['extracted_text'])
            file.write("\n\n")
            file.write(f"--- End of {article['filename']} ---\n")
    print(f"Combined text saved to {output_file}")

# Specify the output file path
output_file_path = "/content/drive/MyDrive/Projects/Deep Learning for NLP/combined_Text.txt"

# Save combined text from all extracted articles
save_combined_text(processed_articles, output_file_path)


### Data Loading

In [ ]:
from datasets import load_dataset, DatasetDict, concatenate_datasets
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments


# Load dataset
dataset1 = load_dataset("poornima9348/finance-alpaca-1k-test")

In [ ]:
import requests
from datasets import Dataset

# URL of the text file hosted on GitHub
file_url = 'https://raw.githubusercontent.com/Satyam-G-Kulkarni/IndustryGPT---Financial-Literacy-and-Personal-Finance-Management/main/combined_Text.txt'

# Fetch the content from the URL
response = requests.get(file_url)
response.raise_for_status()  # Raises an error for failed requests

# Decode the content as text (assuming the text file is in UTF-8 format)
text_data = response.text

# Split the text into entries based on full stops
entries = [sentence.strip() + '.' for sentence in text_data.split('.') if sentence.strip()]

# Prepare the dataset
data_dict = {"text": entries}  # Each entry is a separate sentence
dataset2 = Dataset.from_dict(data_dict)

# Optionally, inspect the dataset
print(dataset2)


In [ ]:
# Combine 'instruction' and 'output' columns into a new 'text' column
def combine_text_columns(example):
    return {'text': f"{example['instruction']} ### {example['output']}"}

# Apply the function to each example in the dataset
dataset1 = dataset1.map(combine_text_columns)

# Remove 'instruction', 'input' and 'output' columns
dataset1['test']=dataset1['test'].remove_columns(['instruction','input', 'output'])

In [ ]:
# Perform the train-test split on the necessary dataset if required
split_dataset1 = dataset1['test'].train_test_split(train_size=0.8)
split_dataset2 = dataset2.train_test_split(test_size=0.2)

In [ ]:
# Concatenate the datasets
merged_train = concatenate_datasets([split_dataset1['train'], split_dataset2['train']])
merged_test = concatenate_datasets([split_dataset1['test'], split_dataset2['test']])

# Create a new DatasetDict with the merged datasets
merged_dataset = DatasetDict({
    'train': merged_train,
    'test': merged_test
})

# Filter out None values in case some splits are missing
merged_dataset = DatasetDict({k: v for k, v in merged_dataset.items() if v is not None})

# Print the merged dataset to verify
print(merged_dataset)

In [ ]:
# Shuffle the dataset and slice it
merged_train_dataset = merged_dataset['train'].shuffle(seed=42).select(range(len(merged_train)))

def transform_conversation(example):
    conversation_text1 = example['text']
    segments = conversation_text1.split('###')

    reformatted_segments = []

    # Iterate over the segments and ensure each segment has a prompt and answer
    for i in range(0, len(segments) - 1, 2):
        prompt = segments[i].strip()
        if i + 1 < len(segments):
            answer = segments[i + 1].strip()
            # Apply the new template
            reformatted_segments.append(f'<s>[INST] {prompt} [/INST] {answer} </s>')
        else:
            # Handle the case where there is no corresponding assistant segment
            reformatted_segments.append(f'<s>[INST] {answer} [/INST] </s>')

    return {'text': ''.join(reformatted_segments)}

# Apply the transformation
transformed_dataset = merged_train_dataset.map(transform_conversation)

In [ ]:
# Shuffle the dataset and slice it
merged_test_dataset = merged_dataset['test'].shuffle(seed=42).select(range(len(merged_test)))

def transform_conversation(example):
    conversation_text1 = example['text']
    segments = conversation_text1.split('###')

    reformatted_segments = []

    # Iterate over the segments and ensure each segment has a prompt and answer
    for i in range(0, len(segments) - 1, 2):
        prompt = segments[i].strip()
        if i + 1 < len(segments):
            answer = segments[i + 1].strip()
            # Apply the new template
            reformatted_segments.append(f'<s>[INST] {prompt} [/INST] {answer} </s>')
        else:
            # Handle the case where there is no corresponding assistant segment
            reformatted_segments.append(f'<s>[INST] {answer} [/INST] </s>')

    return {'text': ''.join(reformatted_segments)}

# Apply the transformation
transformed_test_dataset = merged_test_dataset.map(transform_conversation)

In [ ]:
merged_test_dataset['text'][0]

## **ML Model Building and Training**

### **Base Model**

In [ ]:
# Step 1: Load the GPT-2 model and tokenizer
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# Step 2: Define a function to generate text
import torch

def generate_text(prompt, max_length=100):
    inputs = tokenizer.encode(prompt, return_tensors="pt")
    outputs = model.generate(inputs, max_length=max_length, num_return_sequences=1,
                             no_repeat_ngram_size=2,
                             do_sample=True,
                             top_k=50,
                             top_p=0.95,
                             temperature=1.0)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Step 3: Prompt the model
prompt = "what is finance"
generated_text = generate_text(prompt, max_length=100)

# Display the generated text
print(generated_text)


### **Fine-Tunning of model**

1. Load a llama-2-7b-chat-hf model (chat model)
2. Train it on the our custom dataset from hugging face (1,000 samples), which will produce our fine-tuned model Llama-2-7b-chat-finetune

QLoRA will use a rank of 64 with a scaling parameter of 16. We’ll load the Llama 2 model directly in 4-bit precision using the NF4 type and train it for one epoch

In [ ]:
# The model that you want to train from the Hugging Face hub
model_name = "NousResearch/Llama-2-7b-chat-hf"


# Fine-tuned model name
new_model = "Financial-Literacy-Personal-Finance-chatbot-finetune"

# QLoRA parameters

# LoRA attention dimension
lora_r = 64

# Alpha parameter for LoRA scaling
lora_alpha = 16

# Dropout probability for LoRA layers
lora_dropout = 0.1

# bitsandbytes parameters

# Activate 4-bit precision base model loading
use_4bit = True

# Compute dtype for 4-bit base models
bnb_4bit_compute_dtype = "float16"

# Quantization type (fp4 or nf4)
bnb_4bit_quant_type = "nf4"

# Activate nested quantization for 4-bit base models (double quantization)
use_nested_quant = False

# TrainingArguments parameters

# Output directory where the model predictions and checkpoints will be stored
output_dir = "./results"

# Number of training epochs
num_train_epochs = 1

# Enable fp16/bf16 training (set bf16 to True with an A100)
fp16 = False
bf16 = False

# Batch size per GPU for training
per_device_train_batch_size = 4

# Batch size per GPU for evaluation
per_device_eval_batch_size = 4

# Number of update steps to accumulate the gradients for
gradient_accumulation_steps = 1

# Enable gradient checkpointing
gradient_checkpointing = True

# Maximum gradient normal (gradient clipping)
max_grad_norm = 0.3

# Initial learning rate (AdamW optimizer)
learning_rate = 2e-4

# Weight decay to apply to all layers except bias/LayerNorm weights
weight_decay = 0.001

# Optimizer to use
optim = "paged_adamw_32bit"

# Learning rate schedule
lr_scheduler_type = "cosine"

# Number of training steps (overrides num_train_epochs)
max_steps = -1

# Ratio of steps for a linear warmup (from 0 to learning rate)
warmup_ratio = 0.03

# Group sequences into batches with same length
# Saves memory and speeds up training considerably
group_by_length = True

# Save checkpoint every X updates steps
save_steps = 0

# Log every X updates steps
logging_steps = 25

# SFT parameters

# Maximum sequence length to use
max_seq_length = 350

# Pack multiple short examples in the same input sequence to increase efficiency
packing = False

# Load the entire model on the GPU 0
device_map = {"": 0}

1. First of all, we want to load the dataset we defined. Here, our dataset is already preprocessed but, usually, this is where you would reformat the prompt, filter out bad text, combine multiple datasets, etc.


2. Then, we’re configuring bitsandbytes for 4-bit quantization.


3. Next, we're loading the Llama 2 model in 4-bit precision on a GPU with the corresponding tokenizer.


4. Finally, we're loading configurations for QLoRA, regular training parameters, and passing everything to the SFTTrainer. The training can finally start!

In [ ]:
# Load Reformatted dataset
dataset = transformed_dataset

# Load tokenizer and model with QLoRA configuration
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

# Check GPU compatibility with bfloat16
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16: accelerate training with bf16=True")
        print("=" * 80)

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device_map
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fix weird overflow issue with fp16 training

# Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
)

# Set training parameters
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard"
)

# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=packing,
)

# Train model
trainer.train()

# Specify the path in your Google Drive
new_model_path = '/content/drive/MyDrive/Projects/Deep Learning for NLP/Finance_Chatbot'

# Save the trained model to Google Drive
trainer.model.save_pretrained(new_model_path)


In [ ]:
# Prompting
# Ignore warnings
logging.set_verbosity(logging.CRITICAL)

# Run text generation pipeline with our next model
prompt ="What is mean by financial education?"
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

## **Plots of Tensorboard**

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir results/runs

# **Inference**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, PeftConfig
from transformers import pipeline

def load_model_with_adapters(
    model_name,
    adapter_weights_path,
    device_map='auto',
    use_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    use_nested_quant=True
):
    # Load appropriate compute dtype (either FP16 or BF16)
    compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

    # Set up BitsAndBytes configuration for 4-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=use_4bit,
        bnb_4bit_quant_type=bnb_4bit_quant_type,
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=use_nested_quant,
    )

    # Load the pre-trained model with 4-bit quantization
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map=device_map
    )

    # Load the adapter weights using PeftModel
    model = PeftModel.from_pretrained(model, adapter_weights_path)

    model.config.use_cache = False  # Disable cache for inference

    # Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token  # Align padding
    tokenizer.padding_side = "right"

    return model, tokenizer

# Load the model and tokenizer with adapter weights
model_name = "NousResearch/Llama-2-7b-chat-hf"
adapter_weights_path = "/content/drive/MyDrive/Projects/Deep Learning for NLP/Finance_Chatbot"
model, tokenizer = load_model_with_adapters(model_name, adapter_weights_path)


In [ ]:
def perform_inference(model, tokenizer, input_text):
    # Create a text generation pipeline
    pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=100)

    # Format the prompt with special tokens
    prompt = f"<s>[INST] {input_text} [/INST]"

    # Generate output
    result = pipe(prompt)

    # Extract the generated text from the result
    output_text = result[0]['generated_text']

    return output_text

# Example input
input_text = "What is stock market?"
output_text = perform_inference(model, tokenizer, input_text)

# Print the output
print(output_text)

# **Flask Code**

In [ ]:
import torch
from flask import Flask, request, jsonify
from pyngrok import ngrok
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

# Define the HTML template for the chatbot interface
html_template = """
<!DOCTYPE html>
<html>
<head>
    <title>Chatbot</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            background-color: #f5f5f5;
            margin: 0;
            padding: 0;
            display: flex;
            justify-content: center;
            align-items: center;
            height: 100vh;
        }
        .chat-container {
            width: 80%;
            max-width: 800px;
            background-color: #fff;
            border-radius: 10px;
            box-shadow: 0 0 10px rgba(0, 0, 0, 0.1);
            padding: 20px;
            display: flex;
            flex-direction: column;
            height: 80vh;
        }
        .chat-box {
            flex: 1;
            overflow-y: auto;
            padding: 10px;
            border: 1px solid #ddd;
            border-radius: 5px;
            margin-bottom: 10px;
        }
        .message {
            margin: 10px 0;
            padding: 10px;
            border-radius: 5px;
        }
        .user {
            background-color: #e0f7fa;
            align-self: flex-end;
        }
        .bot {
            background-color: #f1f8e9;
            align-self: flex-start;
        }
        .input-box {
            display: flex;
        }
        .input-box input {
            flex: 1;
            padding: 10px;
            border: 1px solid #ddd;
            border-radius: 5px;
        }
        .input-box button {
            padding: 10px;
            border: none;
            background-color: #007bff;
            color: #fff;
            border-radius: 5px;
            cursor: pointer;
            margin-left: 10px;
        }
        .input-box button:hover {
            background-color: #0056b3;
        }
    </style>
</head>
<body>
    <div class="chat-container">
        <div class="chat-box" id="chat-box"></div>
        <div class="input-box">
            <input type="text" id="user-input" placeholder="Type your message...">
            <button onclick="sendMessage()">Send</button>
        </div>
    </div>

    <script>
        // Function to escape potentially harmful characters in the bot's response
        function escapeHtml(text) {
            const div = document.createElement('div');
            div.innerText = text;
            return div.innerHTML;
        }

        async function sendMessage() {
            const input = document.getElementById('user-input');
            const chatBox = document.getElementById('chat-box');
            const message = input.value.trim();
            if (message === '') return;

            // Add user message to chat box
            chatBox.innerHTML += '<div class="message user">' + escapeHtml(message) + '</div>';
            input.value = '';

            // Send message to server
            const response = await fetch('/chat', {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify({ prompt: message })
            });

            const data = await response.json();

            // Add bot response to chat box (sanitize any HTML)
            chatBox.innerHTML += '<div class="message bot">' + escapeHtml(data.response) + '</div>';
            chatBox.scrollTop = chatBox.scrollHeight;
        }
    </script>
</body>
</html>
"""

# Flask app setup
app = Flask(__name__)

@app.route('/', methods=['GET'])
def home():
    return html_template

@app.route('/chat', methods=['POST'])
def chat():
    # Get the user's prompt from the request
    user_input = request.json.get('prompt')

    # Generate response using the perform_inference function
    bot_response = perform_inference(model, tokenizer, user_input)

    # Return the response as JSON
    return jsonify({"response": bot_response})

if __name__ == '__main__':
    # Start ngrok tunnel
    public_url = ngrok.connect(5000)
    print(" * Running on", public_url)

    # Run Flask app
    app.run()

# **Conclusion**

This project successfully developed a financial chatbot using the LLaMA-2-7b-chat-hf model, fine-tuned on a custom dataset focused on financial literacy and personal finance management. The chatbot generates accurate and relevant responses, effectively conveying complex concepts like financial planning, as shown in its comprehensive explanations.

By employing QLoRA for efficient training in 4-bit precision, the model remains resource-friendly while maintaining a high level of contextual understanding, ensuring users receive reliable advice on investments, retirement, and insurance.

The user-friendly interface, built with Flask and accessible via Ngrok, facilitates seamless interaction, allowing users to engage easily with the chatbot.

Overall, this project highlights the potential of NLP models to empower individuals in their financial decision-making, offering precise and pertinent information. Future enhancements could include expanding the dataset and incorporating personalized recommendations to further elevate the chatbot's effectiveness.